<div align="center">

# Livrable 1 — Classification Automatique d'Images
## Projet DS Leyenda — TouNum

| | |
|:---|:---|
| **Étudiants**  | *BERRETTA BAPTISTE - HIVER ANDREW - KHALED REFKA - GORDIEN ALEXIS* |
| **Promotion**  | *FISA-INFO A5* |
| **École**      | CESI |
| **Date**       | Mai 2026 |
|:---|:---|
</div>

---

### Contexte métier

L'entreprise **TouNum** se spécialise dans la numérisation de documents pour le compte de ses clients.
Dans le cadre de l'automatisation de son pipeline de traitement, il est nécessaire de **trier
automatiquement les images** selon leur nature avant d'appliquer les algorithmes de captioning.

Ce notebook implémente un **réseau de neurones convolutif (CNN)** capable de classifier des images
en **5 catégories** :

| Classe         | Description                             |
|----------------|-----------------------------------------|
| **Painting**   | Peintures et œuvres d'art               |
| **Photo**      | Photographies réelles                   |
| **Schematics** | Schémas, graphes et diagrammes          |
| **Sketch**     | Portraits dessinés en noir et blanc     |
| **Text**       | Documents textuels scannés              |

### Objectif

Entraîner un modèle de **classification automatique** permettant de distinguer les photos des autres
types d'images, avec une attention particulière à la discrimination entre **photos et peintures**
(le cas le plus difficile visuellement).

### Workflow général

```
Données brutes  ──►  Exploration  ──►  Nettoyage  ──►  Prétraitement
                                                              │
           Interface  ◄──  Évaluation  ◄──  Entraînement  ◄───┘
```


---
## Sommaire

| Section | Titre | Description |
|---------|-------|-------------|
| **0** | [Environnement de Travail](#0) | Installation des dependances, imports, configuration globale |
| **1** | [Exploration du Dataset](#1) | Distribution des classes, exemples visuels, dimensions |
| **1b** | [Nettoyage des Images](#1b) | Detection et correction des fichiers corrompus |
| **2** | [Preparation des Donnees](#2) | Labels binaires, augmentation, class weights |
| **3** | [Architecture CNN Binaire](#3) | CNN 4 blocs, Dense(1,sigmoid), binary_crossentropy |
| **4** | [Entrainement](#4) | Callbacks, optimisation, suivi de la convergence |
| **5** | [Analyse des Resultats](#5) | Courbes, AUC-ROC, matrice 2x2, FP/FN |
| **6** | [Biais / Variance](#6) | Diagnostic de l'apprentissage |
| **7** | [Pistes d'Amelioration](#7) | Regularisation avancee, transfer learning |
| **7b** | [Architecture Externe -- fakv8 (binaire)](#7b) | fakv8 en mode binaire, courbes, comparaison |
| **Bonus** | [Classification Multiple](#bonus) | CNN + fakv8 en 5 classes, matrices de confusion |
| **8** | [Interface Interactive](#gui) | Test binaire et multi-classes en conditions reelles |
| **9** | [Conclusion](#10) | Synthese des resultats et perspectives |

> **Comment utiliser ce notebook :**
> Executer les cellules **dans l'ordre** de haut en bas lors de la premiere execution.
> Apres un restart du kernel, utiliser la cellule de *Reprise rapide* (Section 5) pour recharger les modeles sauvegardes.


---
## 0. Environnement de Travail <a id="0"></a>

Avant toute chose, on s'assure que l'environnement d'exécution est correctement configuré :

- **Dépendances** : les librairies nécessaires sont installées si absentes
- **Imports** : toutes les bibliothèques Python sont chargées
- **Configuration** : les hyperparamètres et chemins sont définis en un seul endroit

> Centraliser la configuration dans une seule cellule (`cell-config`) permet de modifier
> les paramètres (taille d'image, batch size, learning rate) sans toucher au code métier.


In [ ]:
# ============================================================
# INSTALLATION DES DÉPENDANCES
# Utilise l'API interne de pip (pas de subprocess) pour installer
# les packages manquants directement dans l'environnement du kernel.
# ============================================================

from pip._internal.cli.main import main as _pip_install

for _pkg in ['seaborn', 'ipywidgets', 'pandas', 'tqdm']:
    try:
        __import__(_pkg)
        print(f"{_pkg} : déjà installé")
    except ImportError:
        print(f"{_pkg} : installation en cours...")
        _pip_install(['install', '--quiet', _pkg])
        print(f"{_pkg} : installé ✓")

print("\nDépendances OK.")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# Reproductibilité : fixer les seeds pour des résultats cohérents entre exécutions
tf.random.set_seed(42)
np.random.seed(42)

# Style des graphiques
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')

# Détection et configuration du GPU
print(f"TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        # memory_growth évite que TF alloue toute la VRAM d'un coup
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU disponible : {gpus[0].name}")
else:
    print("GPU non détecté — entraînement sur CPU (sera plus lent)")

In [ ]:
# ============================================================
# CONFIGURATION GLOBALE
# ============================================================

import os

# --- Résolution du dossier de travail ---
_candidates = ['./data', '/tf/notebooks/data']
DATA_DIR = next((p for p in _candidates if os.path.isdir(p)), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Dossier 'data/' introuvable.\n"
        "Vérifiez que le volume Docker est bien monté avec :\n"
        "  -v \"<dossier_projet>:/tf/notebooks\""
    )

os.chdir(os.path.dirname(os.path.abspath(DATA_DIR)))
DATA_DIR = './data'
print(f"Répertoire de travail : {os.getcwd()}")

# Suppression des dossiers internes vides (ex: _corrupt créé par le nettoyage PIL)
# pour qu'image_dataset_from_directory ne les détecte pas comme des classes.
for _d in os.listdir(DATA_DIR):
    _path = os.path.join(DATA_DIR, _d)
    if _d.startswith('_') and os.path.isdir(_path) and not os.listdir(_path):
        os.rmdir(_path)
        print(f"Dossier vide supprimé : {_path}")

# .h5 à la place de .keras — bug TF 2.14 avec ModelCheckpoint natif
MODEL_SAVE_PATH = './model_classification.h5'

IMG_SIZE       = (128, 128)
BATCH_SIZE     = 32
EPOCHS         = 30
LEARNING_RATE  = 1e-4
VAL_SPLIT      = 0.2   # 80% train, 20% pour val+test (chacun 10%)

# sorted() garantit un ordre alphabétique reproductible :
# Painting=0, Photo=1, Schematics=2, Sketch=3, Text=4
CLASS_NAMES = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
    and not d.startswith('_')
])
NUM_CLASSES = len(CLASS_NAMES)

# Classification binaire : Photo vs tout le reste
PHOTO_CLASS_IDX    = CLASS_NAMES.index('Photo') if 'Photo' in CLASS_NAMES else 1
BINARY_CLASS_NAMES = ['Pas une Photo', 'Photo']

print(f"Classes détectées ({NUM_CLASSES}) : {CLASS_NAMES}")
print(f"Taille des images  : {IMG_SIZE[0]}x{IMG_SIZE[1]} px")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Epochs max         : {EPOCHS}")
print(f"Learning rate      : {LEARNING_RATE}")

---
## 1. Exploration du Dataset

Avant d'entraîner quoi que ce soit, il est essentiel de **comprendre les données** :
- Combien d'images par classe ?
- Y a-t-il un **déséquilibre de classes** (class imbalance) ?
- À quoi ressemblent les images ?

> Un déséquilibre important peut biaiser le modèle vers les classes majoritaires.
> On le compensera plus tard avec des **class weights**.

In [ ]:
# ============================================================
# DISTRIBUTION DES CLASSES
# ============================================================

# Comptage des images pour chaque classe
class_counts = {}
for class_name in CLASS_NAMES:
    class_path = os.path.join(DATA_DIR, class_name)
    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ]
    class_counts[class_name] = len(images)

total = sum(class_counts.values())

# Visualisation en deux graphiques complémentaires
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution du dataset', fontsize=16, fontweight='bold')

colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

# -- Histogramme : permet de voir les valeurs absolues et l'écart entre classes --
bars = ax1.bar(class_counts.keys(), class_counts.values(), color=colors,
               edgecolor='white', linewidth=1.5)
ax1.axhline(total / NUM_CLASSES, color='gray', linestyle='--', alpha=0.8, label='Moyenne')
ax1.set_title("Nombre d'images par classe", fontsize=13)
ax1.set_xlabel('Classe')
ax1.set_ylabel("Nombre d'images")
ax1.legend()
for bar, cnt in zip(bars, class_counts.values()):
    ax1.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 60,
             f'{cnt:,}', ha='center', va='bottom', fontweight='bold')

# -- Camembert : proportions relatives --
ax2.pie(class_counts.values(), labels=class_counts.keys(), colors=colors,
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax2.set_title('Répartition des classes (%)', fontsize=13)

plt.tight_layout()
plt.show()

# Tableau récapitulatif
print(f"\n{'Classe':<15} {'Images':>8} {'%':>7}")
print("-" * 33)
for cls, cnt in class_counts.items():
    flag = ' ' if cnt < total / NUM_CLASSES * 0.5 else ''
    print(f"{cls:<15} {cnt:>8,} {cnt/total*100:>6.1f}%{flag}")
print("-" * 33)
print(f"{'TOTAL':<15} {total:>8,} {'100.0%':>7}")

min_cls = min(class_counts, key=class_counts.get)
max_cls = max(class_counts, key=class_counts.get)
ratio   = class_counts[max_cls] / class_counts[min_cls]
print(f"\nRatio max/min : {ratio:.1f}x — ", end="")
if ratio > 3:
    print(f"déséquilibre significatif ({min_cls} sous-représentée). Class weights nécessaires.")
else:
    print("déséquilibre modéré, gérable.")

In [ ]:
# ============================================================
# EXEMPLES VISUELS PAR CLASSE
# Visualiser quelques images par classe permet de comprendre
# la difficulté de la tâche (ex : peinture réaliste vs photo).
# ============================================================

fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, 3 * NUM_CLASSES))
fig.suptitle("Exemples d'images par classe (5 aléatoires)",
             fontsize=16, fontweight='bold', y=1.01)

for i, class_name in enumerate(CLASS_NAMES):
    class_path = os.path.join(DATA_DIR, class_name)
    all_files  = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    # Sélection reproductible de 5 images
    sample_files = np.random.choice(all_files, min(5, len(all_files)), replace=False)

    for j, img_name in enumerate(sample_files):
        img = tf.keras.utils.load_img(os.path.join(class_path, img_name))
        axes[i, j].imshow(np.array(img))
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(class_name, fontsize=12, fontweight='bold',
                                   rotation=0, labelpad=70, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ANALYSE DES DIMENSIONS (échantillon de 30 images/classe)
# Utile pour choisir la taille de redimensionnement.
# ============================================================

print("Analyse des dimensions d'un échantillon d'images...\n")

records = []
for class_name in CLASS_NAMES:
    class_path = os.path.join(DATA_DIR, class_name)
    all_files  = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    sample     = np.random.choice(all_files, min(30, len(all_files)), replace=False)
    for img_name in sample:
        img = tf.keras.utils.load_img(os.path.join(class_path, img_name))
        w, h = img.size
        records.append({'Classe': class_name, 'Largeur': w, 'Hauteur': h, 'Ratio': w / h})

df_dims = pd.DataFrame(records)

print("Statistiques globales des dimensions (pixels) :")
print(df_dims[['Largeur', 'Hauteur']].describe().round(0).astype(int).to_string())
print(f"\nRatio largeur/hauteur médian : {df_dims['Ratio'].median():.2f}")
print(f"\n  Les images seront redimensionnées en {IMG_SIZE[0]}×{IMG_SIZE[1]} pixels.")
print("    Ce format réduit la mémoire et uniformise les entrées du réseau.")

---
## 1b. Nettoyage des Images Corrompues

Certaines images ont des métadonnées JPEG/PNG malformées que le décodeur TensorFlow (strict) rejette.

**Stratégie :**
1. PIL ouvre chaque image avec `LOAD_TRUNCATED_IMAGES = True` (plus tolérant que TF)
2. Si PIL réussit → re-sauvegarde en RGB propre (supprime les métadonnées corrompues)
3. Si PIL échoue → image irrécupérable, déplacée dans `data/_corrupt/`

> Un fichier `.images_cleaned` est créé après le passage pour éviter de tout re-scanner à chaque exécution.

In [ ]:
# ============================================================
# NETTOYAGE DES IMAGES CORROMPUES
# ============================================================

from PIL import Image, ImageFile
import shutil
from tqdm.notebook import tqdm

FLAG_FILE = '.images_cleaned'

if os.path.exists(FLAG_FILE):
    print("Images déjà nettoyées (flag .images_cleaned présent). Étape ignorée.")
    print("Supprimez ce fichier pour relancer le nettoyage.")
else:
    # PIL tolérant : peut lire les JPEG tronqués que TF refuse
    ImageFile.LOAD_TRUNCATED_IMAGES = True

    corrupt_dir = os.path.join(DATA_DIR, '_corrupt')
    os.makedirs(corrupt_dir, exist_ok=True)

    # Collecte de tous les fichiers image
    all_files = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(DATA_DIR, cls)
        for f in sorted(os.listdir(cls_dir)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                all_files.append(os.path.join(cls_dir, f))

    n_ok = n_fixed = n_corrupt = 0

    for img_path in tqdm(all_files, desc="Nettoyage", unit="img"):
        try:
            with Image.open(img_path) as img:
                img.load()          # Force le décodage complet
                rgb = img.convert('RGB')

            # Re-sauvegarde propre : supprime les métadonnées corrompues
            ext = os.path.splitext(img_path)[1].lower()
            fmt = 'JPEG' if ext in ('.jpg', '.jpeg') else 'PNG'
            rgb.save(img_path, format=fmt, quality=95)
            n_fixed += 1

        except Exception:
            # PIL ne peut pas du tout ouvrir cette image → irrécupérable
            dest = os.path.join(corrupt_dir, os.path.basename(img_path))
            shutil.move(img_path, dest)
            n_corrupt += 1

    # Flag pour éviter de relancer à chaque exécution
    open(FLAG_FILE, 'w').close()

    print(f"\n Images re-sauvegardées proprement : {n_fixed:,}")
    print(f" Images irrécupérables (déplacées vers {corrupt_dir}) : {n_corrupt}")
    if n_corrupt > 0:
        print("   → Ces images sont exclues du dataset automatiquement.")
    print("\nRelancez les cellules de chargement du dataset (cell-load-data) maintenant.")

---
## 2. Preparation des Donnees <a id="2"></a>

### Objectif : Photo ou Pas une Photo ?

La tache est une **classification binaire** : l'image est-elle une photographie reelle,
ou appartient-elle a une autre categorie (Painting, Schematics, Sketch, Text) ?

- **Label 1** -- Photo
- **Label 0** -- Pas une Photo (toutes les autres classes regroupees)

### Strategie de chargement

- **Chargement par batches** avec `image_dataset_from_directory` -- evite de charger ~41 000 images en RAM d'un coup
- **Redimensionnement** a 128x128 px (compromis qualite/vitesse)
- **Normalisation** des pixels `[0, 255]` vers `[0, 1]` (necessaire pour la convergence du reseau)
- **Augmentation** sur le train uniquement (enrichit artificiellement les donnees)
- **Class weights** pour compenser le desequilibre Photo / Non-Photo

### Decoupage train / val / test

```
Dataset complet (~41 399 images)
         |
    +----+----+
   80%       20%
  Train   Val + Test
             |
          +--+--+
         50%   50%
         Val   Test
        (~10%) (~10%)
```

> **Pourquoi un set de test separe ?**  
> La validation sert a guider l'entrainement (early stopping, lr scheduling).  
> Le test, jamais vu pendant l'entrainement, donne une mesure **non biaisee** des performances reelles.


In [ ]:
# ============================================================
# CHARGEMENT DU DATASET
#
# SPLIT val/test : deux instances independantes (shuffle=False)
# pour eviter le data leakage cause par take/skip sur un objet
# partage. Avec le meme seed=42 et shuffle=False, les deux
# instances produisent exactement le meme ordre de fichiers :
#   _vt_1.take(n_val) -> premiere moitie  -> VAL
#   _vt_2.skip(n_val) -> deuxieme moitie -> TEST
# Les deux moities sont garanties disjointes et stables.
# ============================================================

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
)

# Deux instances independantes du meme split validation (shuffle=False)
_vt_1 = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)
_vt_2 = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)

print(f'Classes detectees : {train_ds_raw.class_names}')

n_val_test = tf.data.experimental.cardinality(_vt_1).numpy()
n_val      = n_val_test // 2
n_test     = n_val_test - n_val
val_ds_raw  = _vt_1.take(n_val)   # premiere moitie de l'instance 1
test_ds_raw = _vt_2.skip(n_val)   # deuxieme moitie de l'instance 2

n_train = tf.data.experimental.cardinality(train_ds_raw).numpy()
print(f'Batches  -- Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Images ~ -- Train: {n_train*BATCH_SIZE:,} | Val: {n_val*BATCH_SIZE:,} | Test: {n_test*BATCH_SIZE:,}')


In [ ]:
# ============================================================
# AUGMENTATION DES DONNÉES
# On l'applique UNIQUEMENT sur le set d'entraînement.
#
# Pas de ignore_errors() ici : le nettoyage PIL (cell 07cc88e5)
# a déjà corrigé ou écarté les images corrompues. Sans ignore_errors(),
# la cardinalité du dataset reste CONNUE, ce qui permet à Keras
# d'afficher la progression XX/YY et de recréer l'itérateur
# naturellement à chaque epoch — exactement comme dans le WS.
# ============================================================

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomBrightness(factor=0.15),
    layers.RandomContrast(factor=0.15),
], name="data_augmentation")

rescale  = layers.Rescaling(1.0 / 255.0)
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_train(images, labels):
    images = data_augmentation(images, training=True)
    images = rescale(images)
    return images, labels

def preprocess_eval(images, labels):
    images = rescale(images)
    return images, labels

# Pipeline sans ignore_errors() ni repeat() :
# Keras itère chaque dataset jusqu'à StopIteration (fin naturelle de l'epoch),
# puis recrée un nouvel itérateur pour l'epoch suivante.
train_ds = (train_ds_raw
    .map(preprocess_train, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

val_ds = (val_ds_raw
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

test_ds = (test_ds_raw
    .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

print("Pipeline configuré :")
print("  train_ds  → augmentation + normalisation")
print("  val_ds    → normalisation")
print("  test_ds   → normalisation")

# ============================================================
# REMAPPING BINAIRE : Photo=1, tout le reste=0
# ============================================================
# CLASS_NAMES alphabetique : Painting=0, Photo=1, Schematics=2, Sketch=3, Text=4
# On unifie tout sauf Photo sous le label 0 (Pas une Photo)
# ============================================================

def to_binary(x, y):
    return x, tf.cast(tf.equal(y, PHOTO_CLASS_IDX), tf.int32)

train_ds = train_ds.map(to_binary, num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.map(to_binary,   num_parallel_calls=AUTOTUNE)
test_ds  = test_ds.map(to_binary,  num_parallel_calls=AUTOTUNE)

print('Remapping binaire applique :')
print(f'  Photo (idx {PHOTO_CLASS_IDX}) -> label 1')
print( '  Tout autre           -> label 0 (Pas une Photo)')


In [ ]:
# ============================================================
# VISUALISATION DE L'AUGMENTATION
# On applique l'augmentation sur la même image plusieurs fois
# pour voir la diversité générée.
# ============================================================

# Récupère une image de la classe Photo
photo_dir    = os.path.join(DATA_DIR, 'Photo')
sample_file  = os.listdir(photo_dir)[0]
sample_img   = tf.keras.utils.load_img(os.path.join(photo_dir, sample_file), target_size=IMG_SIZE)
# img_to_array retourne déjà un numpy.ndarray — pas besoin de .numpy()
sample_arr   = tf.keras.utils.img_to_array(sample_img)   # shape (128,128,3), valeurs [0,255]
sample_batch = tf.expand_dims(sample_arr, 0)              # tensor (1,128,128,3) pour le pipeline

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Effet de l'augmentation — même image, 5 variantes",
             fontsize=14, fontweight='bold')

# Ligne 1 : originale (répétée) — sample_arr est un ndarray, .astype suffit
for j in range(5):
    axes[0, j].imshow(sample_arr.astype('uint8'))
    axes[0, j].axis('off')
    axes[0, j].set_title('Original', fontsize=10)

# Ligne 2 : versions augmentées — le résultat de data_augmentation est un tensor TF
for j in range(5):
    aug = data_augmentation(sample_batch, training=True)[0]
    aug = tf.clip_by_value(aug, 0, 255).numpy().astype('uint8')
    axes[1, j].imshow(aug)
    axes[1, j].axis('off')
    axes[1, j].set_title(f'Augmentée #{j+1}', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLASS WEIGHTS — CLASSIFICATION BINAIRE
# ============================================================
# Les poids compensent le desequilibre entre Photo et Non-Photo.
# label 0 = Pas une Photo  (Painting + Schematics + Sketch + Text)
# label 1 = Photo
# ============================================================

from tqdm.notebook import tqdm

# Comptage des 5 classes brutes (avant remapping)
label_counts = {}
for cls in CLASS_NAMES:
    label_counts[cls] = 0

import os as _os
for cls in CLASS_NAMES:
    cls_dir = _os.path.join(DATA_DIR, cls)
    if _os.path.isdir(cls_dir):
        label_counts[cls] = len([f for f in _os.listdir(cls_dir)
                                  if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp'))])

n_photo     = label_counts.get('Photo', 0)
n_not_photo = sum(v for k, v in label_counts.items() if k != 'Photo')
n_total     = n_photo + n_not_photo

# Poids inverses de la frequence
class_weights = {
    0: n_total / (2.0 * n_not_photo) if n_not_photo > 0 else 1.0,
    1: n_total / (2.0 * n_photo)     if n_photo     > 0 else 1.0,
}

print(f"Distribution binaire sur {n_total:,} images :")
print(f"  label 0 = Pas une Photo : {n_not_photo:,} images  (poids {class_weights[0]:.3f})")
print(f"  label 1 = Photo         : {n_photo:,} images  (poids {class_weights[1]:.3f})")
print(f"\nRatio Photo / Total : {n_photo/n_total*100:.1f}%")


---
## 3. Architecture du Reseau de Neurones <a id="3"></a>

### Pourquoi un CNN ?

Les **Reseaux de Neurones Convolutifs (CNN)** sont l'architecture de reference pour la classification d'images :
- Les **filtres de convolution** detectent automatiquement des caracteristiques spatiales (bords, textures, formes)
- Le **partage des poids** rend le modele efficace en parametres
- L'**invariance a la translation** (via MaxPooling) ameliore la generalisation

### Architecture choisie -- CNN binaire a 4 blocs convolutifs

Inspiree de VGGNet, notre architecture empile des blocs `Conv -> BatchNorm -> ReLU -> MaxPool -> Dropout`,
avec une profondeur croissante des filtres (32 -> 64 -> 128 -> 256) :

```
INPUT  (128 x 128 x 3)
  |
  +-> BLOC 1 --- Conv2D(32) x 2 -> BN -> ReLU -> MaxPool(2x2) -> Drop(0.25)
  |              Sortie : 64 x 64 x 32   -- detection : bords, gradients de couleur
  |
  +-> BLOC 2 --- Conv2D(64) x 2 -> BN -> ReLU -> MaxPool(2x2) -> Drop(0.25)
  |              Sortie : 32 x 32 x 64   -- detection : textures, motifs locaux
  |
  +-> BLOC 3 --- Conv2D(128) x 2 -> BN -> ReLU -> MaxPool(2x2) -> Drop(0.25)
  |              Sortie : 16 x 16 x 128  -- detection : formes complexes
  |
  +-> BLOC 4 --- Conv2D(256) -> BN -> ReLU -> MaxPool(2x2) -> Drop(0.25)
  |              Sortie : 8 x 8 x 256    -- semantique globale
  |
  +-> GlobalAveragePooling2D  ->  1 x 256
  |
  +-> Dense(512) -> BN -> ReLU -> Dropout(0.5)
  |
  +-> Dense(1, sigmoid)  ->  P(Photo) dans [0, 1]
           |
       >= 0.5 -> Photo   |   < 0.5 -> Pas une Photo
```

### Justification des composants

| Composant | Role |
|-----------|------|
| **Conv2D** | Extraction de features par filtres apprenants |
| **BatchNormalization** | Stabilise l'entrainement, accelere la convergence |
| **ReLU** | Introduit la non-linearite (sans saturer le gradient) |
| **MaxPooling2D** | Reduit la resolution, renforce l'invariance |
| **Dropout** | Regularisation -- force le reseau a ne pas dependre de neurones specifiques |
| **GlobalAveragePooling** | Alternative plus legere au Flatten -- reduit le risque de surapprentissage |
| **Sigmoid** | Sortie dans [0,1] interpretable directement comme P(Photo) |

### Pourquoi sigmoid et non softmax ?

Avec 2 classes, un seul neurone sigmoid suffit : P(Pas Photo) = 1 - P(Photo).  
Softmax necessiterait 2 neurones, ce qui est equivalent mais plus lourd.  
La loss associee est `binary_crossentropy` (vs `categorical_crossentropy` pour le multiclasse).


In [ ]:
# ============================================================
# CONSTRUCTION DU CNN — CLASSIFICATION BINAIRE
# ============================================================
# Differences vs multiclasse :
#   - Couche de sortie : Dense(1, sigmoid) au lieu de Dense(5, softmax)
#   - sigmoid : retourne P(Photo) dans [0, 1]
#   - Loss : binary_crossentropy (plus adaptee que categorical)
#
# Architecture 4 blocs inchangee :
#   Bloc1 (32) → Bloc2 (64) → Bloc3 (128) → Bloc4 (256) → GAP → Dense(512) → Dense(1)
# ============================================================

def build_cnn_binary(input_shape=(128, 128, 3), dropout_conv=0.25, dropout_dense=0.5):
    """
    CNN 4 blocs pour classification binaire Photo / Pas une Photo.
    Sortie : 1 neurone sigmoid -> P(Photo).
    """
    m = models.Sequential(name='CNN_TouNum_Binaire')
    m.add(layers.Input(shape=input_shape))

    # Bloc 1 : bords et gradients (32 filtres)
    for _ in range(2):
        m.add(layers.Conv2D(32,  (3, 3), padding='same'))
        m.add(layers.BatchNormalization())
        m.add(layers.Activation('relu'))
    m.add(layers.MaxPooling2D((2, 2)))
    m.add(layers.Dropout(dropout_conv))

    # Bloc 2 : textures (64 filtres)
    for _ in range(2):
        m.add(layers.Conv2D(64,  (3, 3), padding='same'))
        m.add(layers.BatchNormalization())
        m.add(layers.Activation('relu'))
    m.add(layers.MaxPooling2D((2, 2)))
    m.add(layers.Dropout(dropout_conv))

    # Bloc 3 : formes complexes (128 filtres)
    for _ in range(2):
        m.add(layers.Conv2D(128, (3, 3), padding='same'))
        m.add(layers.BatchNormalization())
        m.add(layers.Activation('relu'))
    m.add(layers.MaxPooling2D((2, 2)))
    m.add(layers.Dropout(dropout_conv))

    # Bloc 4 : representations semantiques (256 filtres)
    m.add(layers.Conv2D(256, (3, 3), padding='same'))
    m.add(layers.BatchNormalization())
    m.add(layers.Activation('relu'))
    m.add(layers.MaxPooling2D((2, 2)))
    m.add(layers.Dropout(dropout_conv))

    # Classifieur
    m.add(layers.GlobalAveragePooling2D())
    m.add(layers.Dense(512))
    m.add(layers.BatchNormalization())
    m.add(layers.Activation('relu'))
    m.add(layers.Dropout(dropout_dense))

    # Sortie binaire : 1 neurone sigmoid -> P(Photo in [0,1])
    m.add(layers.Dense(1, activation='sigmoid', name='photo_probability'))

    return m


model = build_cnn_binary(input_shape=(*IMG_SIZE, 3))
model.summary()


In [ ]:
# ============================================================
# COMPILATION — CLASSIFICATION BINAIRE
# ============================================================
# Loss : binary_crossentropy
#   Mesuree la distance entre P(Photo) predit et la vraie etiquette 0/1.
#   Equivalente a la cross-entropie categorique avec 2 classes.
#
# Metriques :
#   accuracy  : taux de bonne classification global
#   AUC       : aire sous la courbe ROC (independant du seuil 0.5)
#   Precision : vrais positifs / tous les positifs predits
#   Recall    : vrais positifs / tous les vrais positifs reels
# ============================================================

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ],
)

# Verification : passage d'un batch fictif
dummy = tf.zeros((1, *IMG_SIZE, 3))
out   = model(dummy, training=False)
print(f'Sortie du modele : shape={out.shape}  valeur={out.numpy()[0,0]:.4f} (P(Photo))')
print(f'Interpretation   : 0.0 = certainement PAS une Photo | 1.0 = certainement une Photo')


---
## 4. Entraînement <a id="4"></a>

### Stratégie d'optimisation

L'entraînement utilise l'optimiseur **Adam** (Adaptive Moment Estimation), qui adapte
automatiquement le taux d'apprentissage pour chaque paramètre. C'est le choix standard
pour les CNN car il converge rapidement et tolère bien des hyperparamètres non parfaits.

### Callbacks

Trois mécanismes automatiques pilotent et sécurisent l'entraînement :

| Callback | Paramètres | Rôle |
|----------|-----------|------|
| **EarlyStopping** | `patience=8` | Arrête si `val_loss` ne baisse plus pendant 8 epochs consécutives. Restaure automatiquement les poids du meilleur epoch. |
| **ReduceLROnPlateau** | `patience=3`, `factor=0.5` | Divise le learning rate par 2 après 3 epochs sans amélioration. Affine la convergence quand le modèle est proche de l'optimum. |
| **ModelCheckpoint** | `monitor=val_accuracy` | Sauvegarde le modèle **uniquement** quand `val_accuracy` s'améliore — garantit de conserver le meilleur état. |

### Pourquoi ces valeurs ?

- **patience=8** pour EarlyStopping : suffisamment tolérant pour que ReduceLROnPlateau ait le temps de réduire le LR et relancer la descente.
- **patience=3** pour ReduceLR : déclenche tôt pour ne pas gaspiller d'epochs à un LR trop élevé.
- **factor=0.5** : diviser par 2 est un compromis — pas trop agressif (÷10) ni trop timide (÷1.1).

### Mécanisme anti-surapprentissage intégré

```
Epoch k : val_loss diminue → continue
Epoch k+1..k+3 : val_loss stagne → ReduceLR divise le LR par 2
Epoch k+4..k+8 : val_loss stagne encore → EarlyStopping déclenche
→ Les meilleurs poids (epoch k) sont restaurés automatiquement
```


In [ ]:
# ============================================================
# CALLBACKS
# ============================================================

callbacks = [
    # Arrête l'entraînement si val_loss ne baisse plus pendant 8 epochs
    # → laisse le temps à ReduceLROnPlateau d'agir (patience=3) avant de stopper
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),

    # save_format='h5' forcé explicitement — le format .keras natif de TF 2.14
    # passe un argument 'options' non supporté et lève une ValueError
    keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        save_format='h5',
        verbose=1,
    ),
]

print("Callbacks configurés :")
for cb in callbacks:
    print(f"  - {cb.__class__.__name__}")
print(f"  Modèle sauvegardé dans : {MODEL_SAVE_PATH}")

In [ ]:
# ============================================================
# ENTRAÎNEMENT
#
# Pas de steps_per_epoch ni validation_steps : Keras itère
# chaque dataset jusqu'à épuisement naturel (comme dans le WS).
# La cardinalité est connue → progression XX/YY affichée.
# ============================================================

print("Début de l'entraînement...")
print(f"  Taille image   : {IMG_SIZE}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Epochs max     : {EPOCHS}")
print(f"  Learning rate  : {LEARNING_RATE}")
print("-" * 50)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nEntraînement terminé. Meilleur modèle sauvegardé : {MODEL_SAVE_PATH}")

In [ ]:
# ============================================================
# SAUVEGARDE HISTORIQUE — CNN Binaire
# ============================================================
# L'objet history est perdu au prochain restart du kernel.
# On serialise history.history (dict Python) dans un JSON
# pour pouvoir recharger les courbes sans reentrainer.
# ============================================================

import json as _json

_HIST_PATH = './history_cnn.json'
_hist_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}

with open(_HIST_PATH, 'w', encoding='utf-8') as _f:
    _json.dump(_hist_data, _f, indent=1)

_n_epochs = len(next(iter(_hist_data.values())))
print(f'Historique CNN Binaire sauvegarde : {_HIST_PATH}')
print(f'  {_n_epochs} epochs | metriques : {list(_hist_data.keys())}')

# Meilleure val_accuracy ou val_loss selon dispo
_best_key = 'val_accuracy' if 'val_accuracy' in _hist_data else 'val_loss'
_best_val = max(_hist_data[_best_key]) if 'accuracy' in _best_key else min(_hist_data[_best_key])
print(f'  Meilleur {_best_key} : {_best_val:.4f}')


---
## 5. Analyse des Resultats <a id="5"></a>

### Courbes d'apprentissage

Les courbes **train vs validation** sont l'outil principal pour diagnostiquer l'etat de l'apprentissage.
On les lit en cherchant trois patterns :

```
  Accuracy
     |
  1  +-- train  ------------------
     |   val    -----------------   <- Bon equilibre : courbes proches et hautes
     |
     |   train  ------------------
     |   val    --------           <- Surapprentissage : ecart croissant
     |
     |   train  ------
     |   val    ------             <- Sous-apprentissage : les deux sont basses
     |
  0  +----------------------------> Epochs
```

### Metriques specifiques a la classification binaire

| Metrique | Formule | Interpretation |
|----------|---------|---------------|
| **Accuracy** | (TP+TN) / N | Taux global de bonne classification |
| **AUC-ROC** | Aire sous la courbe ROC | Discriminabilite independante du seuil -- 1.0 = parfait, 0.5 = aleatoire |
| **Precision** | TP / (TP + FP) | Parmi les images predites Photo, combien le sont vraiment ? |
| **Rappel** | TP / (TP + FN) | Parmi les vraies Photos, combien sont detectees ? |
| **F1-score** | 2 x P x R / (P + R) | Equilibre precision/rappel -- utile si classes desequilibrees |

### Matrice de confusion binaire 2x2

```
                  Predit
                  +--------------+------------+
                  | Pas une Photo|   Photo    |
         +--------+--------------+------------+
  Reel   |Pas Photo|   TN        |    FP      |
         +--------+--------------+------------+
         |  Photo |   FN        |    TP      |
         +--------+--------------+------------+
```

- **TN** : Non-photo correctement rejetee
- **TP** : Photo correctement detectee
- **FP** : Fausse alarme -- non-photo classee Photo
- **FN** : Photo manquee -- non detectee


---
### Reprise après restart du kernel

> **Note :** Si le kernel a été redémarré (arrêt du conteneur Docker, fermeture du navigateur, etc.),
> les variables Python sont perdues mais le modèle sauvegardé sur disque persiste.
> La cellule ci-dessous recharge le modèle directement depuis `model_classification.h5`
> — inutile de ré-entraîner.


In [ ]:
# ============================================================
# REPRISE RAPIDE -- apres un restart du container Docker
#
# Recharge le modele CNN principal + tous les historiques JSON.
# Executer dans l'ordre : cell-imports -> cell-config ->
# cell-load-data -> cell-augmentation -> CETTE cellule.
# ============================================================

import json as _json, types as _types

# --- Variables de chemins pour la section Bonus ---
CNN_MULTI_SAVE_PATH = './model_cnn_multiclass.h5'
FAK_MULTI_SAVE_PATH = './fakv8_multiclass_model.h5'

# --- Modele CNN principal ---
if os.path.exists(MODEL_SAVE_PATH):
    model = keras.models.load_model(MODEL_SAVE_PATH)
    print(f'Modele CNN binaire charge : {MODEL_SAVE_PATH}')
    print(f'  {model.input_shape} -> {model.output_shape}')
else:
    print(f'Modele CNN non trouve : {MODEL_SAVE_PATH}')

# --- Rechargement des historiques ---
def _load_history(path, label):
    if os.path.exists(path):
        with open(path, encoding='utf-8') as _f:
            data = _json.load(_f)
        n  = len(next(iter(data.values())))
        bk = 'val_accuracy' if 'val_accuracy' in data else 'val_loss'
        bv = max(data[bk]) if 'accuracy' in bk else min(data[bk])
        print(f'  {label} : {n} ep | best {bk} = {bv:.4f}')
        return _types.SimpleNamespace(history=data)
    print(f'  {label} : non trouve ({path})')
    return None

print()
print('Historiques :')
history          = _load_history('./history_cnn.json',       'CNN Binaire')
history_fak      = _load_history('./history_fak.json',       'fakv8 Binaire')
history_cnn_multi= _load_history('./history_cnn_multi.json', 'CNN Multi')
history_fak_multi= _load_history('./history_fak_multi.json', 'fakv8 Multi')

print()
print('Reprise prete.')
print('  plot_history(history)           -> courbes CNN binaire')
print('  plot_history(history_fak)       -> courbes fakv8 binaire')
print('  plot_history(history_cnn_multi) -> courbes CNN multi')
print('  plot_history(history_fak_multi) -> courbes fakv8 multi')


In [ ]:
# ============================================================
# COURBES D'ENTRAÎNEMENT
# ============================================================

def plot_history(history):
    """Affiche les courbes loss et accuracy (train vs validation)."""
    epochs_ran = range(1, len(history.history['loss']) + 1)

    fig, (ax2, ax1) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Évolution de l'entraînement", fontsize=15, fontweight='bold')

    # -- Courbe Accuracy --
    ax2.plot(epochs_ran, history.history['accuracy'],     'b-o', ms=4, lw=2, label='Train')
    ax2.plot(epochs_ran, history.history['val_accuracy'], 'r-o', ms=4, lw=2, label='Validation')
    best_acc_epoch = int(np.argmax(history.history['val_accuracy'])) + 1
    best_acc       = max(history.history['val_accuracy'])
    ax2.scatter([best_acc_epoch], [best_acc], color='green', zorder=5, s=100,
                label=f'Best val_acc ({best_acc:.3f})')
    ax2.set_title('Précision (Accuracy)', fontsize=13)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1.05)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # -- Courbe Loss --
    ax1.plot(epochs_ran, history.history['loss'],     'b-o', ms=4, lw=2, label='Train')
    ax1.plot(epochs_ran, history.history['val_loss'], 'r-o', ms=4, lw=2, label='Validation')
    best_epoch = int(np.argmin(history.history['val_loss'])) + 1
    ax1.axvline(best_epoch, color='green', ls='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
    ax1.set_title('Fonction de perte (Loss)', fontsize=13)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Binary Crossentropy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)


    plt.tight_layout()
    plt.show()

    # Résumé chiffré
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc   = history.history['val_accuracy'][-1]
    gap             = final_train_acc - final_val_acc

    print(f"\n📊 Résumé :")
    print(f"  Epochs effectuées  : {len(epochs_ran)}")
    print(f"  Meilleure val_acc  : {best_acc:.4f} ({best_acc*100:.2f}%) à l'epoch {best_acc_epoch}")
    print(f"  Dernière train_acc : {final_train_acc:.4f}")
    print(f"  Dernière val_acc   : {final_val_acc:.4f}")
    print(f"  Écart train/val    : {gap:.4f}", end="  ")
    if gap > 0.15:
        print("→   Surapprentissage probable")
    elif final_val_acc < 0.70:
        print("→   Sous-apprentissage probable")
    else:
        print("→  Apprentissage stable")


plot_history(history)

In [ ]:
# ============================================================
# EVALUATION FINALE — CLASSIFICATION BINAIRE
# ============================================================

import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve)
import matplotlib.pyplot as plt

y_true_all = []
y_prob_all = []

for x_batch, y_batch in test_ds:
    probs = model.predict(x_batch, verbose=0)[:, 0]  # P(Photo)
    y_prob_all.extend(probs.tolist())
    y_true_all.extend(y_batch.numpy().tolist())

y_true = np.array(y_true_all, dtype=int)
y_prob = np.array(y_prob_all)
y_pred = (y_prob >= 0.5).astype(int)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
auc  = roc_auc_score(y_true, y_prob)

print('=' * 55)
print('RESULTATS — Notre CNN Binaire (set de test)')
print('=' * 55)
print(f'  Accuracy   : {acc*100:.2f}%   (global)')
print(f'  AUC        : {auc:.4f}   (robuste au seuil)')
print(f'  Precision  : {prec*100:.2f}%   (parmi pred. Photo, vrais Photos)')
print(f'  Recall     : {rec*100:.2f}%   (Photos reelles detectees)')
print(f'  F1 Score   : {f1:.4f}')
print('=' * 55)

# Courbe ROC
fpr, tpr, _ = roc_curve(y_true, y_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC={auc:.3f})')
ax.plot([0,1], [0,1], 'k--', alpha=0.4, label='Aleatoire')
ax.fill_between(fpr, tpr, alpha=0.1)
ax.set_xlabel('Taux de Faux Positifs'); ax.set_ylabel('Taux de Vrais Positifs')
ax.set_title('Courbe ROC — Notre CNN Binaire', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('AUC=1.0 = parfait | AUC=0.5 = aleatoire')


In [ ]:
# ============================================================
# MATRICE DE CONFUSION — BINAIRE 2x2
# ============================================================
# TN : Pas Photo predit | vraie etiquette Pas Photo
# FP : Photo predit     | vraie etiquette Pas Photo  (fausse alarme)
# FN : Pas Photo predit | vraie etiquette Photo      (manque)
# TP : Photo predit     | vraie etiquette Photo      (detection correcte)
# ============================================================

import matplotlib.pyplot as plt

cm = np.zeros((2, 2), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[int(t), int(p)] += 1

bin_labels = ['Pas une Photo', 'Photo']
corner_labels = {(0,0): 'TN', (0,1): 'FP', (1,0): 'FN', (1,1): 'TP'}

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        c = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, f'{cm[i,j]}\n({cm[i,j]/cm.sum()*100:.1f}%)',
                ha='center', va='center', fontsize=14, fontweight='bold', color=c)
        ax.text(j + 0.42, i - 0.38, corner_labels[(i, j)],
                ha='right', va='top', fontsize=9, color='gray', style='italic')

ax.set_xticks([0, 1]); ax.set_xticklabels(bin_labels, fontsize=12)
ax.set_yticks([0, 1]); ax.set_yticklabels(bin_labels, fontsize=12)
ax.set_xlabel('Prediction', fontsize=13); ax.set_ylabel('Verite terrain', fontsize=13)
ax.set_title('Matrice de Confusion — Binaire', fontsize=14, fontweight='bold')
plt.colorbar(im)
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
print(f'TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print(f'Faux positifs : {fp/(fp+tn)*100:.1f}% des Non-Photos classees comme Photo')
print(f'Faux negatifs : {fn/(fn+tp)*100:.1f}% des vraies Photos non detectees')


In [ ]:
# ============================================================
# EXEMPLES D'ERREURS — FAUX POSITIFS ET FAUX NEGATIFS
# ============================================================
# Faux Positifs (FP) : images classees 'Photo' alors qu'elles ne le sont pas
#   -> comprendre ce que le modele confond avec une photo
# Faux Negatifs (FN) : vraies Photos non detectees
#   -> comprendre quelles photos sont difficiles a reconnaitre
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

err_images, err_true, err_prob = [], [], []
for x_batch, y_batch in test_ds:
    probs = model.predict(x_batch, verbose=0)[:, 0]
    err_images.extend(x_batch.numpy())
    err_true.extend(y_batch.numpy().tolist())
    err_prob.extend(probs.tolist())

err_images = np.array(err_images)
err_true   = np.array(err_true, dtype=int)
err_prob   = np.array(err_prob)
err_pred   = (err_prob >= 0.5).astype(int)

fp_idx = np.where((err_pred == 1) & (err_true == 0))[0]  # Faux Positifs
fn_idx = np.where((err_pred == 0) & (err_true == 1))[0]  # Faux Negatifs

print(f'Faux Positifs : {len(fp_idx)} (non-Photos classees Photo)')
print(f'Faux Negatifs : {len(fn_idx)} (Photos non detectees)')

def show_errors(indices, title, label, n_max=5):
    idxs = indices[:n_max]
    if len(idxs) == 0:
        print(f'Aucune erreur de type {title}'); return
    fig, axes = plt.subplots(1, len(idxs), figsize=(3 * len(idxs), 3))
    if len(idxs) == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=13, fontweight='bold', color='red')
    for ax, i in zip(axes, idxs):
        ax.imshow(np.clip(err_images[i], 0, 1))
        ax.set_title(f'P(Photo)={err_prob[i]:.2f}', fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()

show_errors(fp_idx, 'FAUX POSITIFS — Classees Photo par erreur', 'FP')
show_errors(fn_idx, 'FAUX NEGATIFS — Photos non detectees',      'FN')


---
## 6. Analyse Biais / Variance

Le dilemme **biais/variance** est central en Machine Learning. Il décrit le compromis entre deux types d'erreurs opposées.

### 📉 Sous-apprentissage (Biais élevé)

Le modèle est **trop simple** pour capturer les patterns des données.

- **Symptômes** : accuracy d'entraînement **et** de validation toutes deux faibles
- **Causes** : architecture trop petite, trop peu d'epochs, learning rate inadapté
- **Solutions** : augmenter la capacité du modèle, entraîner plus longtemps

### 📈 Surapprentissage (Variance élevée)

Le modèle **mémorise** les données d'entraînement au lieu d'apprendre des patterns généraux.

- **Symptômes** : accuracy train élevée, accuracy validation bien inférieure (grand écart)
- **Causes** : modèle trop grand, données insuffisantes, absence de régularisation
- **Solutions** : dropout, data augmentation, early stopping, régularisation L2

### 📊 Lecture des courbes d'apprentissage

```
 Accuracy
    │
 1  │    train ──────────────────────────
    │    val   ────────────────────────── ← bon équilibre : courbes proches 
    │
    │    train ────────────────────
    │    val   ─────────                  ← surapprentissage : écart croissant
    │
 0  │    train ─────                      ← sous-apprentissage : les deux basses
    │    val   ─────
    └─────────────────────────────────► Epochs
```

### Dans notre modèle

Plusieurs mécanismes limitent le surapprentissage :

| Mécanisme | Effet |
|-----------|-------|
| **Dropout (0.25 / 0.5)** | Force le réseau à apprendre des représentations redondantes |
| **BatchNormalization** | Régularise implicitement en réduisant la dépendance entre couches |
| **Data augmentation** | Chaque image est légèrement différente à chaque epoch |
| **EarlyStopping** | Arrête avant la phase de mémorisation |
| **Class weights** | Évite que le modèle « triche » en ignorant les classes rares |

---
## 7. Pistes d'Amélioration

### Techniques de régularisation supplémentaires

| Technique | Description | Utilisation |
|-----------|-------------|-------------|
| **Régularisation L2** | Pénalise les poids trop grands dans les couches Dense | `Dense(512, kernel_regularizer=l2(1e-4))` |
| **Label Smoothing** | Atténue la confiance du modèle sur les labels | `SparseCategoricalCrossentropy(label_smoothing=0.1)` |
| **Mixup / CutMix** | Mélange deux images pendant l'entraînement | Augmentation avancée |

### Architectures alternatives

Le **Transfer Learning** exploite des réseaux pré-entraînés sur ImageNet (1,4M images, 1000 classes).
Les couches basses ont déjà appris des features génériques (bords, textures) — on ré-entraîne
uniquement les couches hautes sur notre dataset :

```python
# Exemple avec MobileNetV2 (léger, performant)
base = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3),
                                          include_top=False, weights='imagenet')
base.trainable = False   # Gèle les couches pré-entraînées

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
```

Gain attendu : **+10 à +20% d'accuracy** avec moins d'epochs d'entraînement.

### Amélioration des données

- **Oversampling de Sketch** : générer plus d'images via augmentation agressive
- **Collecte de données** : ajouter plus d'images dans les classes difficiles (peintures réalistes)
- **Nettoyage** : vérifier que chaque image est bien dans la bonne classe (labelling errors)

---
## 7b. Comparaison -- Architecture Externe fakv8 (mode binaire) <a id="7b"></a>

Cette section entraine **fakv8** sur la **meme tache binaire** (Photo / Pas une Photo)
afin de comparer les deux approches a conditions egales.

### Analyse des differences architecturales

| Critere | Notre CNN Binaire | fakv8 Binaire |
|---------|-----------------|---------------|
| **Taille images** | 128x128 px | **256x256 px** (4x plus de pixels) |
| **Batch size** | 32 | 16 |
| **Bloc 4** | Conv(256) x 1 | **Conv(256) x 2** (plus profond) |
| **Dropout convolutif** | 0.25 (fixe) | **0.2->0.2->0.3->0.4** (progressif) |
| **Learning rate** | 1e-4 | **3e-4** (plus eleve) |
| **EarlyStopping patience** | 8 epochs | 8 epochs |
| **Sortie** | Dense(1, sigmoid) | Dense(1, sigmoid) |
| **Loss** | binary_crossentropy | binary_crossentropy |

### Prediction

**fakv8 devrait gagner +1 a +4%** grace a :
1. **Images 256x256** : plus de detail pour discriminer les textures de peinture vs photos
2. **Bloc 4 plus profond** : 2 convolutions -> representations plus riches
3. **Dropout progressif** : regularisation plus forte dans les couches profondes

**Contreparties** : entrainement ~2x plus lent, plus de VRAM.

> **Note :** Un seul modele fakv8 est entraine ici (fakv8_test). Il n'y a pas de modele pre-entraine
> multiclasse utilise dans cette section -- la comparaison est entierement binaire et reproductible.


In [ ]:
# ============================================================
# DONNEES POUR FAKV8 -- chargement a 256x256 + remapping binaire
# Split val/test : deux instances independantes (meme fix que cell-load-data).
# ============================================================

FAKV8_IMG_SIZE  = (256, 256)
FAKV8_BATCH     = 16
FAKV8_SAVE_PATH = './fakv8_test_model.h5'

train_ds_fak_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='training',
    seed=42, image_size=FAKV8_IMG_SIZE, batch_size=FAKV8_BATCH,
    label_mode='int', shuffle=True,
)
_vt_fak_1 = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='validation',
    seed=42, image_size=FAKV8_IMG_SIZE, batch_size=FAKV8_BATCH,
    label_mode='int', shuffle=False,
)
_vt_fak_2 = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='validation',
    seed=42, image_size=FAKV8_IMG_SIZE, batch_size=FAKV8_BATCH,
    label_mode='int', shuffle=False,
)

n_vt_fak       = tf.data.experimental.cardinality(_vt_fak_1).numpy()
n_val_fak      = n_vt_fak // 2
val_ds_fak_raw  = _vt_fak_1.take(n_val_fak)
test_ds_fak_raw = _vt_fak_2.skip(n_val_fak)
n_tr_fak = tf.data.experimental.cardinality(train_ds_fak_raw).numpy()
print(f'Dataset 256x256 -- Train: {n_tr_fak} batches | Val: {n_val_fak}')

# Augmentation
rescale_fak  = layers.Rescaling(1.0 / 255.0)
data_aug_fak = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name='aug_fakv8')

def preprocess_fak_train(x, y):
    x = data_aug_fak(x, training=True)
    x = rescale_fak(x)
    return x, y

def preprocess_fak_eval(x, y):
    return rescale_fak(x), y

def to_binary_fak(x, y):
    return x, tf.cast(tf.equal(y, PHOTO_CLASS_IDX), tf.int32)

AUTOTUNE_FAK = tf.data.AUTOTUNE

train_ds_fak = (
    train_ds_fak_raw
    .map(preprocess_fak_train, num_parallel_calls=AUTOTUNE_FAK)
    .map(to_binary_fak, num_parallel_calls=AUTOTUNE_FAK)
    .prefetch(AUTOTUNE_FAK)
)
val_ds_fak = (
    val_ds_fak_raw
    .map(preprocess_fak_eval, num_parallel_calls=AUTOTUNE_FAK)
    .map(to_binary_fak, num_parallel_calls=AUTOTUNE_FAK)
    .prefetch(AUTOTUNE_FAK)
)
test_ds_fak = (
    test_ds_fak_raw
    .map(preprocess_fak_eval, num_parallel_calls=AUTOTUNE_FAK)
    .map(to_binary_fak, num_parallel_calls=AUTOTUNE_FAK)
    .prefetch(AUTOTUNE_FAK)
)

print('Pipeline fakv8 binaire pret.')
print(f'  Photo (idx {PHOTO_CLASS_IDX}) -> label 1')
print( '  Tout autre           -> label 0 (Pas une Photo)')


In [ ]:
# ============================================================
# ARCHITECTURE fakv8 — CLASSIFICATION BINAIRE
# ============================================================
# Differences vs notre CNN :
#   - Input 256x256 (4x plus de pixels)
#   - Bloc 4 : 2 convolutions au lieu de 1
#   - Dropout progressif par bloc (0.2->0.2->0.3->0.4)
#   - Sortie : Dense(1, sigmoid) = meme paradigme binaire que notre CNN
# ============================================================

def build_fakv8_binary(input_shape=(256, 256, 3)):
    """Architecture fakv8 adaptee a la classification binaire Photo/Pas Photo."""
    m = models.Sequential(name='CNN_fakv8_Binaire')
    m.add(layers.Input(shape=input_shape))

    # Bloc 1 : 32 filtres
    m.add(layers.Conv2D(32,  (3,3), activation='relu', padding='same'))
    m.add(layers.Conv2D(32,  (3,3), activation='relu', padding='same'))
    m.add(layers.MaxPooling2D((2,2)))
    m.add(layers.Dropout(0.2))

    # Bloc 2 : 64 filtres
    m.add(layers.Conv2D(64,  (3,3), activation='relu', padding='same'))
    m.add(layers.Conv2D(64,  (3,3), activation='relu', padding='same'))
    m.add(layers.MaxPooling2D((2,2)))
    m.add(layers.Dropout(0.2))

    # Bloc 3 : 128 filtres
    m.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
    m.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
    m.add(layers.MaxPooling2D((2,2)))
    m.add(layers.Dropout(0.3))

    # Bloc 4 : 256 filtres (2 convolutions)
    m.add(layers.Conv2D(256, (3,3), activation='relu', padding='same'))
    m.add(layers.Conv2D(256, (3,3), activation='relu', padding='same'))
    m.add(layers.MaxPooling2D((2,2)))
    m.add(layers.Dropout(0.4))

    # Classifieur
    m.add(layers.GlobalAveragePooling2D())
    m.add(layers.Dense(512, activation='relu'))
    m.add(layers.Dropout(0.5))

    # Sortie binaire
    m.add(layers.Dense(1, activation='sigmoid', name='photo_probability'))

    return m


model_fak = build_fakv8_binary(input_shape=(*FAKV8_IMG_SIZE, 3))

model_fak.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ],
)
model_fak.summary(line_length=80)
print(f'\nParametres : {model_fak.count_params():,}')


In [ ]:
# ============================================================
# ENTRAINEMENT FAKV8 BINAIRE
# ============================================================

os.makedirs('.', exist_ok=True)

callbacks_fak = [
    keras.callbacks.ModelCheckpoint(
        filepath=FAKV8_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        save_format='h5',
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
]

print('Entrainement fakv8 (binaire Photo/Pas Photo)...')
print(f'  Image size : {FAKV8_IMG_SIZE}  |  Batch : {FAKV8_BATCH}  |  LR : 3e-4')
print(f'  Sauvegarde : {FAKV8_SAVE_PATH}')
print('-' * 50)

history_fak = model_fak.fit(
    train_ds_fak,
    epochs=EPOCHS,
    validation_data=val_ds_fak,
    class_weight=class_weights,
    callbacks=callbacks_fak,
    verbose=1,
)

print(f'\nMeilleur fakv8 binaire sauvegarde : {FAKV8_SAVE_PATH}')


In [ ]:
# ============================================================
# SAUVEGARDE HISTORIQUE — fakv8 Binaire
# ============================================================
# L'objet history est perdu au prochain restart du kernel.
# On serialise history.history (dict Python) dans un JSON
# pour pouvoir recharger les courbes sans reentrainer.
# ============================================================

import json as _json

_HIST_PATH = './history_fak.json'
_hist_data = {k: [float(v) for v in vals] for k, vals in history_fak.history.items()}

with open(_HIST_PATH, 'w', encoding='utf-8') as _f:
    _json.dump(_hist_data, _f, indent=1)

_n_epochs = len(next(iter(_hist_data.values())))
print(f'Historique fakv8 Binaire sauvegarde : {_HIST_PATH}')
print(f'  {_n_epochs} epochs | metriques : {list(_hist_data.keys())}')

# Meilleure val_accuracy ou val_loss selon dispo
_best_key = 'val_accuracy' if 'val_accuracy' in _hist_data else 'val_loss'
_best_val = max(_hist_data[_best_key]) if 'accuracy' in _best_key else min(_hist_data[_best_key])
print(f'  Meilleur {_best_key} : {_best_val:.4f}')


In [ ]:
# ============================================================
# COURBES D'ENTRAINEMENT -- fakv8 Binaire
# ============================================================
# Meme visualisation que pour notre CNN (Section 5).
# plot_history() est defini dans cell-curves.
# ============================================================

print('=== Courbes d entrainement : fakv8 Binaire ===')
plot_history(history_fak)


In [ ]:
# ============================================================
# COMPARAISON BINAIRE : Notre CNN vs fakv8
# ============================================================
# Meme tache (Photo / Pas une Photo), meme loss (binary_crossentropy)
# Seule difference : taille d'entree et architecture
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score

# Rechargement des meilleurs checkpoints
model_best     = keras.models.load_model(MODEL_SAVE_PATH,  compile=False)
model_fak_best = keras.models.load_model(FAKV8_SAVE_PATH,  compile=False)

def binary_metrics(mdl, ds):
    """Retourne (accuracy, AUC) sur un dataset."""
    y_true_l, y_prob_l = [], []
    for xb, yb in ds:
        probs = mdl.predict(xb, verbose=0)[:, 0]
        y_prob_l.extend(probs.tolist())
        y_true_l.extend(yb.numpy().tolist())
    yt = np.array(y_true_l, dtype=int)
    yp = np.array(y_prob_l)
    acc = accuracy_score(yt, (yp >= 0.5).astype(int))
    auc = roc_auc_score(yt, yp)
    return acc, auc

acc_cnn, auc_cnn = binary_metrics(model_best,     test_ds)
acc_fak, auc_fak = binary_metrics(model_fak_best, test_ds_fak)

# Tableau recapitulatif
print('=' * 68)
print(f'{"Modele":<30} {"Input":>10} {"Params":>12} {"Accuracy":>10} {"AUC":>8}')
print('-' * 68)
print(f'{"Notre CNN Binaire":<30} {"128x128":>10} {model_best.count_params():>12,} {acc_cnn*100:>9.2f}% {auc_cnn:>8.4f}')
print(f'{"fakv8 Binaire":<30} {"256x256":>10} {model_fak_best.count_params():>12,} {acc_fak*100:>9.2f}% {auc_fak:>8.4f}')
print('=' * 68)

# Graphiques comparatifs
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparaison Binaire — Notre CNN vs fakv8', fontsize=14, fontweight='bold')

models_names  = ['Notre CNN (128x128)', 'fakv8 (256x256)']
colors_bar    = ['#3498db', '#e74c3c']

# Accuracy
axes[0].bar(models_names, [acc_cnn*100, acc_fak*100], color=colors_bar)
for i, v in enumerate([acc_cnn*100, acc_fak*100]):
    axes[0].text(i, v + 0.3, f'{v:.2f}%', ha='center', fontweight='bold')
axes[0].set_ylim(0, 100); axes[0].set_title('Accuracy (%)'); axes[0].set_ylabel('%')

# AUC
axes[1].bar(models_names, [auc_cnn, auc_fak], color=colors_bar)
for i, v in enumerate([auc_cnn, auc_fak]):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')
axes[1].set_ylim(0, 1); axes[1].set_title('AUC (ROC)')

# Params
params = [model_best.count_params()/1e6, model_fak_best.count_params()/1e6]
axes[2].bar(models_names, params, color=colors_bar)
for i, v in enumerate(params):
    axes[2].text(i, v + 0.01, f'{v:.1f}M', ha='center', fontweight='bold')
axes[2].set_title('Parametres (M)')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


---
## Bonus. Classification Multiple -- 5 categories <a id="bonus"></a>

Les sections precedentes font de la **classification binaire** (Photo / Pas une Photo).
Cette section bonus entraine les **memes architectures** (Notre CNN + fakv8) en mode
**multi-classes** : les 5 categories originales du dataset TouNum.

| Label | Classe |
|-------|--------|
| 0 | Painting |
| 1 | Photo |
| 2 | Schematics |
| 3 | Sketch |
| 4 | Text |

**Differences vs binaire :**
- Couche de sortie : `Dense(5, softmax)` au lieu de `Dense(1, sigmoid)`
- Loss : `sparse_categorical_crossentropy` au lieu de `binary_crossentropy`
- Metriques : accuracy par classe, matrice de confusion 5x5
- Class weights : 5 poids (un par classe) au lieu de 2


In [ ]:
# ============================================================
# DONNEES MULTICLASSE -- 5 classes, sans remapping binaire
# Split val/test : deux instances independantes (meme fix).
# ============================================================

CNN_MULTI_SAVE_PATH = './model_cnn_multiclass.h5'
FAK_MULTI_SAVE_PATH = './fakv8_multiclass_model.h5'

MULTI_BATCH     = 32
MULTI_FAK_BATCH = 16
MULTI_FAK_SIZE  = (256, 256)

# --- Class weights 5 classes ---
class_weights_multi = {}
for _i, _cls in enumerate(CLASS_NAMES):
    _n = label_counts.get(_cls, 0)
    class_weights_multi[_i] = (n_total / (NUM_CLASSES * _n)) if _n > 0 else 1.0
print('Class weights multiclasse :')
for _i, _cls in enumerate(CLASS_NAMES):
    print(f'  [{_i}] {_cls:<15} : {class_weights_multi[_i]:.3f}')

# --- Preprocessing 128x128 ---
rescale_m  = layers.Rescaling(1.0/255.0)
data_aug_m = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.1),
], name='aug_multi')

def preprocess_multi_train(x, y): return rescale_m(data_aug_m(x, training=True)), y
def preprocess_multi_eval(x, y):  return rescale_m(x), y

_AM = tf.data.AUTOTUNE

# Train 128x128
train_ds_multi_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='training',
    seed=42, image_size=IMG_SIZE, batch_size=MULTI_BATCH,
    label_mode='int', shuffle=True,
)
# Val/Test 128x128 : deux instances independantes
_vt_m_1 = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=MULTI_BATCH,
    label_mode='int', shuffle=False,
)
_vt_m_2 = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=MULTI_BATCH,
    label_mode='int', shuffle=False,
)
_nv_m = tf.data.experimental.cardinality(_vt_m_1).numpy() // 2
val_ds_multi   = _vt_m_1.take(_nv_m).map(preprocess_multi_eval,  _AM).prefetch(_AM)
test_ds_multi  = _vt_m_2.skip(_nv_m).map(preprocess_multi_eval,  _AM).prefetch(_AM)
train_ds_multi = train_ds_multi_raw.map(preprocess_multi_train, _AM).prefetch(_AM)
print(f'CNN 128x128 -- Train: {tf.data.experimental.cardinality(train_ds_multi_raw)} batches | Val: {_nv_m}')

# --- Preprocessing 256x256 ---
rescale_mf  = layers.Rescaling(1.0/255.0)
data_aug_mf = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name='aug_multi_fak')

def preprocess_mf_train(x, y): return rescale_mf(data_aug_mf(x, training=True)), y
def preprocess_mf_eval(x, y):  return rescale_mf(x), y

# Train 256x256
train_ds_multi_fak_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='training',
    seed=42, image_size=MULTI_FAK_SIZE, batch_size=MULTI_FAK_BATCH,
    label_mode='int', shuffle=True,
)
# Val/Test 256x256 : deux instances independantes
_vt_mf_1 = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='validation',
    seed=42, image_size=MULTI_FAK_SIZE, batch_size=MULTI_FAK_BATCH,
    label_mode='int', shuffle=False,
)
_vt_mf_2 = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VAL_SPLIT, subset='validation',
    seed=42, image_size=MULTI_FAK_SIZE, batch_size=MULTI_FAK_BATCH,
    label_mode='int', shuffle=False,
)
_nv_mf = tf.data.experimental.cardinality(_vt_mf_1).numpy() // 2
val_ds_multi_fak   = _vt_mf_1.take(_nv_mf).map(preprocess_mf_eval,  _AM).prefetch(_AM)
test_ds_multi_fak  = _vt_mf_2.skip(_nv_mf).map(preprocess_mf_eval,  _AM).prefetch(_AM)
train_ds_multi_fak = train_ds_multi_fak_raw.map(preprocess_mf_train, _AM).prefetch(_AM)
print(f'fakv8 256x256 -- Train: {tf.data.experimental.cardinality(train_ds_multi_fak_raw)} batches | Val: {_nv_mf}')
print('Pipelines multiclasse prets (5 classes, sans remapping binaire).')


In [ ]:
# ============================================================
# ARCHITECTURE CNN -- CLASSIFICATION MULTIPLE (5 classes)
# ============================================================
# Identique a build_cnn_binary() mais :
#   - Dense(NUM_CLASSES, softmax) au lieu de Dense(1, sigmoid)
#   - sparse_categorical_crossentropy comme loss
# ============================================================

def build_cnn_multiclass(input_shape=(128, 128, 3),
                          num_classes=NUM_CLASSES,
                          dropout_conv=0.25, dropout_dense=0.5):
    m = models.Sequential(name='CNN_TouNum_Multi')
    m.add(layers.Input(shape=input_shape))

    for filters in [32, 64, 128]:
        for _ in range(2):
            m.add(layers.Conv2D(filters, (3,3), padding='same'))
            m.add(layers.BatchNormalization())
            m.add(layers.Activation('relu'))
        m.add(layers.MaxPooling2D((2,2)))
        m.add(layers.Dropout(dropout_conv))

    m.add(layers.Conv2D(256, (3,3), padding='same'))
    m.add(layers.BatchNormalization())
    m.add(layers.Activation('relu'))
    m.add(layers.MaxPooling2D((2,2)))
    m.add(layers.Dropout(dropout_conv))

    m.add(layers.GlobalAveragePooling2D())
    m.add(layers.Dense(512))
    m.add(layers.BatchNormalization())
    m.add(layers.Activation('relu'))
    m.add(layers.Dropout(dropout_dense))

    # Sortie 5 classes
    m.add(layers.Dense(num_classes, activation='softmax', name='class_probabilities'))
    return m


model_cnn_multi = build_cnn_multiclass(input_shape=(*IMG_SIZE, 3))

model_cnn_multi.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model_cnn_multi.summary()
print(f'\nParametres : {model_cnn_multi.count_params():,}')
print(f'Sortie     : {model_cnn_multi.output_shape} -- softmax 5 classes')


In [ ]:
# ============================================================
# ENTRAINEMENT CNN MULTICLASSE
# ============================================================

cb_cnn_multi = [
    keras.callbacks.ModelCheckpoint(
        filepath=CNN_MULTI_SAVE_PATH, monitor='val_accuracy',
        save_best_only=True, save_format='h5', verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=4, min_lr=1e-6, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1,
    ),
]

print('Entrainement CNN Multi-classes (5 classes)...')
print(f'  Image size : {IMG_SIZE}  Batch : {MULTI_BATCH}  LR : {LEARNING_RATE}')
print(f'  Sauvegarde : {CNN_MULTI_SAVE_PATH}')
print('-' * 55)

history_cnn_multi = model_cnn_multi.fit(
    train_ds_multi,
    epochs=EPOCHS,
    validation_data=val_ds_multi,
    class_weight=class_weights_multi,
    callbacks=cb_cnn_multi,
    verbose=1,
)

_best = max(history_cnn_multi.history['val_accuracy'])
print(f'\nMeilleure val_accuracy CNN Multi : {_best:.4f} ({_best*100:.2f}%)')
print(f'Modele sauvegarde : {CNN_MULTI_SAVE_PATH}')


In [ ]:
import json as _json
_d = {k: [float(v) for v in vals] for k, vals in history_cnn_multi.history.items()}
with open('./history_cnn_multi.json', 'w', encoding='utf-8') as _f:
    _json.dump(_d, _f, indent=1)
_n = len(next(iter(_d.values())))
_bv = max(_d['val_accuracy'])
print(f'Historique CNN Multi sauvegarde : ./history_cnn_multi.json')
print(f'  {_n} epochs | meilleure val_accuracy : {_bv:.4f} ({_bv*100:.2f}%)')


In [ ]:
print('=== Courbes CNN Multi-classes ===')
plot_history(history_cnn_multi)


In [ ]:
# ============================================================
# ARCHITECTURE fakv8 -- CLASSIFICATION MULTIPLE (5 classes)
# ============================================================

def build_fakv8_multiclass(input_shape=(256, 256, 3), num_classes=NUM_CLASSES):
    m = models.Sequential(name='CNN_fakv8_Multi')
    m.add(layers.Input(shape=input_shape))

    for filters, drop in [(32, 0.2), (64, 0.2), (128, 0.3)]:
        m.add(layers.Conv2D(filters, (3,3), activation='relu', padding='same'))
        m.add(layers.Conv2D(filters, (3,3), activation='relu', padding='same'))
        m.add(layers.MaxPooling2D((2,2)))
        m.add(layers.Dropout(drop))

    # Bloc 4 : 256 filtres, 2 convolutions
    m.add(layers.Conv2D(256, (3,3), activation='relu', padding='same'))
    m.add(layers.Conv2D(256, (3,3), activation='relu', padding='same'))
    m.add(layers.MaxPooling2D((2,2)))
    m.add(layers.Dropout(0.4))

    m.add(layers.GlobalAveragePooling2D())
    m.add(layers.Dense(512, activation='relu'))
    m.add(layers.Dropout(0.5))

    # Sortie 5 classes
    m.add(layers.Dense(num_classes, activation='softmax', name='class_probabilities'))
    return m


model_fak_multi = build_fakv8_multiclass(input_shape=(*MULTI_FAK_SIZE, 3))

model_fak_multi.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model_fak_multi.summary(line_length=80)
print(f'\nParametres : {model_fak_multi.count_params():,}')
print(f'Sortie     : {model_fak_multi.output_shape} -- softmax 5 classes')


In [ ]:
# ============================================================
# ENTRAINEMENT fakv8 MULTICLASSE
# ============================================================

cb_fak_multi = [
    keras.callbacks.ModelCheckpoint(
        filepath=FAK_MULTI_SAVE_PATH, monitor='val_accuracy',
        save_best_only=True, save_format='h5', verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=4, min_lr=1e-6, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1,
    ),
]

print('Entrainement fakv8 Multi-classes (5 classes)...')
print(f'  Image size : {MULTI_FAK_SIZE}  Batch : {MULTI_FAK_BATCH}  LR : 3e-4')
print(f'  Sauvegarde : {FAK_MULTI_SAVE_PATH}')
print('-' * 55)

history_fak_multi = model_fak_multi.fit(
    train_ds_multi_fak,
    epochs=EPOCHS,
    validation_data=val_ds_multi_fak,
    class_weight=class_weights_multi,
    callbacks=cb_fak_multi,
    verbose=1,
)

_best = max(history_fak_multi.history['val_accuracy'])
print(f'\nMeilleure val_accuracy fakv8 Multi : {_best:.4f} ({_best*100:.2f}%)')
print(f'Modele sauvegarde : {FAK_MULTI_SAVE_PATH}')


In [ ]:
import json as _json
_d = {k: [float(v) for v in vals] for k, vals in history_fak_multi.history.items()}
with open('./history_fak_multi.json', 'w', encoding='utf-8') as _f:
    _json.dump(_d, _f, indent=1)
_n = len(next(iter(_d.values())))
_bv = max(_d['val_accuracy'])
print(f'Historique fakv8 Multi sauvegarde : ./history_fak_multi.json')
print(f'  {_n} epochs | meilleure val_accuracy : {_bv:.4f} ({_bv*100:.2f}%)')


In [ ]:
print('=== Courbes fakv8 Multi-classes ===')
plot_history(history_fak_multi)


In [ ]:
# ============================================================
# COMPARAISON MULTICLASSE : Notre CNN vs fakv8
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Rechargement des meilleurs checkpoints
_cnn_m = keras.models.load_model(CNN_MULTI_SAVE_PATH, compile=False)
_fak_m = keras.models.load_model(FAK_MULTI_SAVE_PATH, compile=False)

def eval_multiclass(mdl, ds):
    y_true_l, y_pred_l = [], []
    for xb, yb in ds:
        preds = np.argmax(mdl.predict(xb, verbose=0), axis=1)
        y_pred_l.extend(preds.tolist())
        y_true_l.extend(yb.numpy().tolist())
    yt = np.array(y_true_l, dtype=int)
    yp = np.array(y_pred_l, dtype=int)
    acc = np.mean(yt == yp)
    cm  = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for t, p in zip(yt, yp): cm[t, p] += 1
    return acc, cm

acc_cnn_m, cm_cnn = eval_multiclass(_cnn_m, test_ds_multi)
acc_fak_m, cm_fak = eval_multiclass(_fak_m, test_ds_multi_fak)

# --- Tableau recap ---
print('=' * 68)
print(f'{"Modele":<30} {"Input":>8} {"Params":>12} {"Accuracy":>10}')
print('-' * 68)
print(f'{"Notre CNN Multi":30} {"128x128":>8} {_cnn_m.count_params():>12,} {acc_cnn_m*100:>9.2f}%')
print(f'{"fakv8 Multi":30} {"256x256":>8} {_fak_m.count_params():>12,} {acc_fak_m*100:>9.2f}%')
print('=' * 68)

# --- Matrices de confusion ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Matrices de Confusion Multi-classes', fontsize=14, fontweight='bold')

for ax, cm, title in [
    (axes[0], cm_cnn, f'Notre CNN Multi ({acc_cnn_m*100:.1f}%)'),
    (axes[1], cm_fak, f'fakv8 Multi ({acc_fak_m*100:.1f}%)'),
]:
    im = ax.imshow(cm, cmap='Blues')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            c = 'white' if cm[i,j] > cm.max()/2 else 'black'
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    fontsize=10, fontweight='bold', color=c)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=25, ha='right')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Prediction', fontsize=11); ax.set_ylabel('Verite terrain', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax)

plt.tight_layout(); plt.show()

# --- Bar chart accuracy ---
fig2, ax2 = plt.subplots(figsize=(7, 4))
names_m = ['Notre CNN (128x128)', 'fakv8 (256x256)']
accs_m  = [acc_cnn_m*100, acc_fak_m*100]
bars = ax2.bar(names_m, accs_m, color=['#3498db', '#e74c3c'], width=0.5)
for bar, v in zip(bars, accs_m):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.3,
             f'{v:.2f}%', ha='center', fontweight='bold')
ax2.set_ylim(0, 100); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Comparaison Accuracy -- Classification Multiple', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


---
## 9. Interface Graphique Interactive <a id="9"></a>

L'interface ci-dessous permet de tester les modeles entraines sur n'importe quelle image,
avec un **choix entre classification binaire et multi-classes**.

### Deux modes de classification

| Mode | Modeles disponibles | Sortie |
|------|---------------------|--------|
| **Binaire** | Notre CNN + fakv8 | Photo / Pas une Photo + P(Photo) |
| **Multi-classes** | EfficientNetB0 | 5 categories (Painting, Photo, Schematics, Sketch, Text) |

### Fonctionnement

1. **Choisissez le mode** : Binaire (Photo/Pas Photo) ou Multi-classes (5 categories)
2. **Selectionnez un modele** dans la liste deroulante correspondante
3. **Uploadez une image** depuis votre disque
4. Cliquez sur **Analyser**
5. L'image s'affiche avec la **classe predite**, le **score de confiance** (%), et la distribution des probabilites

### Interpretation des resultats

| Confiance | Interpretation |
|-----------|---------------|
| >= 75% | Prediction fiable -- le modele est sur de sa reponse |
| 50-75% | Prediction incertaine -- la classe est probable mais d'autres sont possibles |
| < 50% | Image ambigue ou hors distribution -- a verifier manuellement |

> Les modeles chargent automatiquement la bonne resolution selon leur architecture.
> Le mode Multi-classes utilise EfficientNetB0 avec rescaling interne [0,1] vers [0,255].


In [ ]:
# ============================================================
# INTERFACE GRAPHIQUE -- BINAIRE ET MULTI-CLASSES
# ============================================================
# Mode BINAIRE    : Notre CNN (128x128) + fakv8 (256x256)
#                   Photo / Pas une Photo
# Mode MULTICLASSE: Notre CNN multi (128x128) + fakv8 multi (256x256)
#                   5 categories : Painting, Photo, Schematics, Sketch, Text
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np, io, os
from PIL import Image as PIL_Image

_FAKV8_BIN_PATH  = './fakv8_test_model.h5'
_CNN_MULTI_PATH  = CNN_MULTI_SAVE_PATH   # './model_cnn_multiclass.h5'
_FAK_MULTI_PATH  = FAK_MULTI_SAVE_PATH   # './fakv8_multiclass_model.h5'

# Catalogues par mode
_BINARY_CATALOG = {
    f'Notre CNN Binaire (128x128) -- {MODEL_SAVE_PATH}' : MODEL_SAVE_PATH,
    f'fakv8 Binaire (256x256) -- {_FAKV8_BIN_PATH}'    : _FAKV8_BIN_PATH,
}
_MULTI_CATALOG = {
    f'Notre CNN Multi (128x128) -- {_CNN_MULTI_PATH}'  : _CNN_MULTI_PATH,
    f'fakv8 Multi (256x256) -- {_FAK_MULTI_PATH}'      : _FAK_MULTI_PATH,
}

def _available(cat):
    return {k: v for k, v in cat.items() if os.path.exists(v)}

# --- Widgets ---
mode_toggle = widgets.ToggleButtons(
    options=['Binaire -- Photo / Pas une Photo', 'Multi-classes -- 5 categories'],
    description='Mode :',
    button_style='info',
    style={'description_width': 'initial'},
)
model_dropdown = widgets.Dropdown(
    options=list(_available(_BINARY_CATALOG).keys()) or ['(aucun modele disponible)'],
    description='Modele :',
    layout=widgets.Layout(width='70%'),
    style={'description_width': 'initial'},
)
upload_btn = widgets.FileUpload(accept='image/*', multiple=False, description='Image')
run_btn    = widgets.Button(description='Analyser', button_style='success', icon='search')
out_area   = widgets.Output()

# --- Etat global ---
_active_model     = None
_active_model_key = None
_active_mode      = 'binary'

def _load_model_gui(label, catalog):
    global _active_model, _active_model_key
    if label == _active_model_key and _active_model is not None:
        return
    path = catalog[label]
    try:
        _active_model     = keras.models.load_model(path, compile=False)
        _active_model_key = label
        print(f'Modele charge : {path}')
    except Exception as e:
        print(f'Erreur chargement : {e}')

def on_mode_change(change):
    global _active_mode, _active_model, _active_model_key
    _active_model = None; _active_model_key = None
    if 'Binaire' in change['new']:
        _active_mode = 'binary'
        avail = _available(_BINARY_CATALOG)
    else:
        _active_mode = 'multi'
        avail = _available(_MULTI_CATALOG)
    model_dropdown.options = list(avail.keys()) if avail else ['(aucun modele disponible)']

def on_run(b):
    with out_area:
        clear_output(wait=True)
        if not upload_btn.value:
            print('Veuillez uploader une image.'); return

        # Lecture image
        fi = (list(upload_btn.value.values())[0]
              if isinstance(upload_btn.value, dict) else upload_btn.value[0])
        content = (fi.get('content', b'') if isinstance(fi, dict)
                   else bytes(getattr(fi, 'content', b'')))
        try:
            img_pil = PIL_Image.open(io.BytesIO(content)).convert('RGB')
        except Exception as e:
            print(f'Erreur lecture image : {e}'); return

        # Chargement modele
        catalog = _BINARY_CATALOG if _active_mode == 'binary' else _MULTI_CATALOG
        avail   = _available(catalog)
        sel     = model_dropdown.value
        if sel not in avail:
            print(f'Modele non disponible : {sel}'); return
        _load_model_gui(sel, avail)
        if _active_model is None: return

        # Preprocessing
        model_img_size = tuple(_active_model.input_shape[1:3])
        img_r  = img_pil.resize(model_img_size)
        arr    = np.array(img_r, dtype=np.float32) / 255.0
        arr    = arr[np.newaxis, ...]
        raw_out = _active_model.predict(arr, verbose=0)[0]

        # Interpretation
        if _active_mode == 'binary':
            p_photo    = float(raw_out[0])
            probs_dict = {'Pas une Photo': 1.0-p_photo, 'Photo': p_photo}
            best_class = 'Photo' if p_photo >= 0.5 else 'Pas une Photo'
            best_conf  = (p_photo if p_photo >= 0.5 else 1.0-p_photo) * 100
        else:
            probs_dict = {c: float(p) for c, p in zip(CLASS_NAMES, raw_out)}
            best_idx   = int(np.argmax(raw_out))
            best_class = CLASS_NAMES[best_idx]
            best_conf  = float(raw_out[best_idx]) * 100

        # Affichage
        import matplotlib.pyplot as plt
        mode_str = 'Binaire' if _active_mode == 'binary' else 'Multi-classes'
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle(f'Mode {mode_str} -- {sel.split(" -- ")[0]}', fontsize=11, color='gray')
        ax1.imshow(np.array(img_r) / 255.0)
        ax1.axis('off')
        ax1.set_title(f'Prediction : {best_class}\nConfiance  : {best_conf:.1f}%',
                      fontsize=13, fontweight='bold',
                      color='green' if best_conf > 75 else 'orange')
        names = list(probs_dict.keys())
        vals  = [probs_dict[n]*100 for n in names]
        cols  = ['#2ecc71' if n == best_class else '#3498db' for n in names]
        bars  = ax2.barh(names, vals, color=cols, edgecolor='white', height=0.5)
        for bar, v in zip(bars, vals):
            ax2.text(min(v+1, 92), bar.get_y()+bar.get_height()/2,
                     f'{v:.1f}%', va='center', fontweight='bold', fontsize=10)
        ax2.set_xlim(0, 100); ax2.set_xlabel('Probabilite (%)')
        ax2.set_title('Scores de confiance')
        ax2.axvline(50, color='gray', ls='--', alpha=0.5)
        ax2.grid(axis='x', alpha=0.3)
        plt.tight_layout(); plt.show()

mode_toggle.observe(on_mode_change, names='value')
run_btn.on_click(on_run)

avail_bin   = _available(_BINARY_CATALOG)
avail_multi = _available(_MULTI_CATALOG)
if not avail_bin and not avail_multi:
    print('Aucun modele .h5 trouve. Entrainez d abord les modeles.')
else:
    display(widgets.VBox([
        widgets.HTML('<h3>Interface de Classification -- TouNum</h3>'),
        mode_toggle, model_dropdown,
        widgets.HTML('<hr style="margin:8px 0">'),
        upload_btn, run_btn, out_area,
    ]))
    print(f'Modeles binaires disponibles     : {len(avail_bin)}/2')
    print(f'Modeles multi-classes disponibles: {len(avail_multi)}/2')


---
## 10. Conclusion <a id="10"></a>

### Synthese de la demarche

Ce livrable a mis en oeuvre un pipeline complet de classification d'images, de l'exploration
des donnees brutes jusqu'au deploiement d'une interface de test interactive :

| Etape | Action | Resultat |
|-------|--------|----------|
| **Exploration** | Analyse de la distribution et des dimensions | Desequilibre detecte -- class weights necessaires |
| **Nettoyage** | Correction des JPEG corrompus via PIL | Dataset propre, sans erreur de decodage TF |
| **Binary remap** | Photo=1 / tout le reste=0 | Tache simplifiee, donnees mieux equilibrees |
| **Augmentation** | Flip, rotation, zoom, brightness | Generalisation amelioree, overfitting reduit |
| **Architecture** | CNN binaire 4 blocs (32->64->128->256) | ~720K parametres, Dense(1,sigmoid) |
| **Entrainement** | Adam + binary_crossentropy + EarlyStopping | Convergence automatique, meilleurs poids sauvegardes |
| **Evaluation** | Matrice 2x2 + AUC-ROC + F1 | Mesure non biaisee sur set de test |
| **Comparaison** | fakv8 binaire (256x256) | Impact de la resolution sur la discrimination |
| **Transfer Learning** | EfficientNetB0 (5 classes, 2 phases) | +10 a +20% vs CNN from scratch |

### Reponse a la problematique

Le modele est capable de **detecter automatiquement les photographies** dans un corpus mixte
(Photo vs Painting/Schematics/Sketch/Text) avec une precision elevee.  
La classification binaire est plus robuste et plus interpretable que le multi-classes pour
cette tache de tri de premier niveau.

### Comparatif des approches

| Modele | Tache | Architecture | Avantage |
|--------|-------|-------------|----------|
| **Notre CNN Binaire** | Photo / Pas Photo | 4 blocs, 128x128 | Leger, rapide, entrainable en 10 min |
| **fakv8 Binaire** | Photo / Pas Photo | 4 blocs, 256x256 | Plus de detail, meilleure precision attendue |
| **EfficientNetB0** | 5 classes | MBConv pre-entraine, 224x224 | SOTA transfer learning, ~95%+ attendu |

### Ce livrable dans le contexte TouNum

La classification automatique constitue la **premiere brique** du pipeline de traitement :
les images sont triees par type avant d'etre envoyees vers les algorithmes specialises
(OCR pour Text, description artistique pour Painting, etc.).  
Le mode binaire Photo / Non-Photo permet une **validation rapide** du contenu photographique.

### Pistes d'amelioration non explorees

- **Ensemble CNN + EfficientNetB0** -- combiner les deux pour reduire l'erreur
- **Grad-CAM** -- visualiser les zones de l'image qui influencent la decision
- **Augmentation avancee** (CutMix, Mixup) -- meilleure generalisation sur les cas limites
- **Distillation** -- comprimer EfficientNetB0 dans un modele plus leger pour le deploiement
